# 03 建模 - Walk-Forward验证
LightGBM vs Ridge，Walk-Forward滚动验证，特征重要性分析

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.models import WalkForwardValidator, predict_scores
from src.clustering import cluster_stocks
from src.utils import calc_sharpe, calc_max_drawdown

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
df = pd.read_parquet('../data/processed_data.parquet')

# 聚类特征
df = cluster_stocks(df, n_clusters=4)
df['cluster_feat'] = df['cluster'].fillna(-1).astype(float)

feature_cols = ['factor_pe', 'factor_momentum_20', 'factor_vol_60', 'cluster_feat']
target_col = 'fwd_ret_5d'

print(f'数据准备完成: {df.shape}')

In [ ]:
# Walk-Forward 验证
validator = WalkForwardValidator(
    train_window=504,  # ~2年
    test_window=63,    # ~3个月
    mode='sliding'
)

results = {}

for model_type in ['lgb', 'ridge']:
    print(f'\n=== {model_type.upper()} ===')
    res = validator.validate(df, feature_cols=feature_cols, target_col=target_col, model_type=model_type)
    results[model_type] = res
    print(f'总IC: {res["overall_ic"]:.4f}')
    print(f'窗口数: {len(res["window_metrics"])}')
    
    if res['feature_importances'] is not None:
        print('\n特征重要性:')
        for name, imp in zip(feature_cols, res['feature_importances']):
            print(f'  {name}: {imp:.4f}')

In [ ]:
# 各窗口 IC 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, res) in zip(axes, results.items()):
    wm = res['window_metrics']
    if not wm.empty:
        ax.plot(wm['window'], wm['test_ic'], marker='o', linestyle='-', label=f'{name.upper()} IC')
        ax.axhline(y=res['overall_ic'], color='red', linestyle='--', label=f'mean IC={res["overall_ic"]:.4f}')
    ax.set_xlabel('Window')
    ax.set_ylabel('Test IC')
    ax.set_title(f'{name.upper()} Walk-Forward IC')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 特征重要性对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, res) in zip(axes, results.items()):
    if res['feature_importances'] is not None:
        cols = res['feature_cols']
        imp = res['feature_importances']
        sorted_idx = np.argsort(imp)
        ax.barh([cols[i] for i in sorted_idx], [imp[i] for i in sorted_idx])
        ax.set_title(f'{name.upper()} - 特征重要性')
        ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# 预测值 vs 真实值散点图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(res['y_true'], res['y_pred'], alpha=0.3, s=1)
    ax.plot([-0.2, 0.2], [-0.2, 0.2], 'r--', alpha=0.5)
    ax.set_xlabel('True Return')
    ax.set_ylabel('Predicted Return')
    ax.set_title(f'{name.upper()} (IC={res["overall_ic"]:.4f})')

plt.tight_layout()
plt.show()